# Plant disease classification with a custom CNN

This notebook is an executable example for the PlantVillage image-folder layout. It does not include data or claim metrics until the user runs it with a local dataset. Set `PLANTVILLAGE_ROOT` before running.


In [ ]:
import hashlib
import os
import random
from pathlib import Path

# Set seeds before importing TensorFlow for more repeatable runs.
SEED = int(os.environ.get("PLANT_DISEASE_SEED", "42"))
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")
random.seed(SEED)

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from tensorflow.keras import Model
from tensorflow.keras import backend as K
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (Activation, BatchNormalization, Conv2D, Dense,
                                     Dropout, Flatten, Input, LeakyReLU, MaxPooling2D)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array

tf.random.set_seed(SEED)


In [ ]:
DEFAULT_IMAGE_SIZE = (256, 256)
MAX_IMAGES_PER_CLASS = int(os.environ.get("PLANT_DISEASE_MAX_IMAGES_PER_CLASS", "100"))
DATA_ROOT = Path(os.environ.get("PLANTVILLAGE_ROOT", "data/PlantVillage")).expanduser()
OUTPUT_DIR = Path(os.environ.get("PLANT_DISEASE_OUTPUT_DIR", "outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def resolve_class_root(root: Path) -> Path:
    """Return a directory whose immediate children are class directories."""
    candidates = [
        root / "train",
        root / "New Plant Diseases Dataset(Augmented)" / "New Plant Diseases Dataset(Augmented)" / "train",
        root,
    ]
    for candidate in candidates:
        if candidate.is_dir() and any(child.is_dir() for child in candidate.iterdir()):
            return candidate
    expected = " or ".join(str(path) for path in candidates)
    raise FileNotFoundError(
        "PlantVillage data was not found. Set PLANTVILLAGE_ROOT to the extracted dataset "
        f"directory; checked: {expected}"
    )

CLASS_ROOT = resolve_class_root(DATA_ROOT)
print(f"Using class folders from: {CLASS_ROOT.resolve()}")


In [ ]:
def load_images(class_root: Path, max_images_per_class: int):
    images, labels, image_paths, skipped = [], [], [], []
    class_dirs = sorted(path for path in class_root.iterdir() if path.is_dir())
    if not class_dirs:
        raise ValueError(f"No class directories found below {class_root}")
    for class_dir in class_dirs:
        paths = sorted(path for path in class_dir.iterdir()
                       if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
        for image_path in paths[:max_images_per_class]:
            image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
            if image is None:
                skipped.append(str(image_path))
                continue
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, DEFAULT_IMAGE_SIZE, interpolation=cv2.INTER_AREA)
            images.append(img_to_array(image))
            labels.append(class_dir.name)
            image_paths.append(image_path)
    if not images:
        raise ValueError(f"No readable images found below {class_root}")
    if skipped:
        print(f"Skipped {len(skipped)} unreadable image(s)")
    return np.asarray(images, dtype=np.float32) / 255.0, np.asarray(labels), image_paths

X, labels_text, image_paths = load_images(CLASS_ROOT, MAX_IMAGES_PER_CLASS)
# Exact duplicate files can otherwise cross a random split and inflate validation.
digests = {}
for image_path in image_paths:
    digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
    if digest in digests:
        raise ValueError(f"Duplicate image content detected: {image_path} and {digests[digest]}")
    digests[digest] = image_path
label_encoder = LabelEncoder()
labels_int = label_encoder.fit_transform(labels_text)
class_names = list(label_encoder.classes_)
if len(class_names) < 2:
    raise ValueError("At least two classes are required for classification")
if np.min(np.bincount(labels_int)) < 2:
    raise ValueError("Each class needs at least two readable images for a stratified split")
y = tf.keras.utils.to_categorical(labels_int, num_classes=len(class_names))
print(f"Loaded {len(X)} images across {len(class_names)} classes")


In [ ]:
# The holdout is a validation split, not an independent test set.
X_train, X_valid, y_train, y_valid, labels_train, labels_valid = train_test_split(
    X, y, labels_int, test_size=0.20, random_state=SEED, stratify=labels_int
)
print(f"Train: {len(X_train)} | validation: {len(X_valid)}")

augment = ImageDataGenerator(
    rotation_range=25, width_shift_range=0.1, height_shift_range=0.1,
    shear_range=0.2, zoom_range=0.2, horizontal_flip=True, fill_mode="nearest"
)


In [ ]:
EPOCHS = int(os.environ.get("PLANT_DISEASE_EPOCHS", "10"))
BATCH_SIZE = int(os.environ.get("PLANT_DISEASE_BATCH_SIZE", "32"))
HEIGHT, WIDTH, DEPTH = 256, 256, 3
LEARNING_RATE = 1e-3
input_shape = (HEIGHT, WIDTH, DEPTH)

model = Sequential([
    Input(shape=input_shape),
    Conv2D(32, (3, 3), padding="same"), LeakyReLU(alpha=0.1),
    BatchNormalization(), MaxPooling2D(pool_size=(3, 3)), Dropout(0.25),
    Conv2D(64, (3, 3), padding="same"), LeakyReLU(alpha=0.1),
    BatchNormalization(), Conv2D(64, (3, 3), padding="same"),
    LeakyReLU(alpha=0.1), BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)), Dropout(0.25),
    Conv2D(128, (3, 3), padding="same"), LeakyReLU(alpha=0.1),
    BatchNormalization(), Conv2D(128, (3, 3), padding="same"),
    LeakyReLU(alpha=0.1), BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)), Dropout(0.25),
    Flatten(), Dense(1024, name="feature_dense"), LeakyReLU(alpha=0.1),
    BatchNormalization(), Dropout(0.5), Dense(len(class_names), activation="softmax"),
], name="plant_disease_cnn")
model.summary()


In [ ]:
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
history = model.fit(
    augment.flow(X_train, y_train, batch_size=BATCH_SIZE, seed=SEED),
    validation_data=(X_valid, y_valid),
    epochs=EPOCHS,
    verbose=1,
)
validation_metrics = model.evaluate(X_valid, y_valid, verbose=0, return_dict=True)
print("Validation metrics:", validation_metrics)
model.save(OUTPUT_DIR / "custom_cnn.keras")


In [ ]:
epochs = range(1, len(history.history["accuracy"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, history.history["accuracy"], label="train")
axes[0].plot(epochs, history.history["val_accuracy"], label="validation")
axes[0].set(title="Accuracy", xlabel="Epoch"); axes[0].legend()
axes[1].plot(epochs, history.history["loss"], label="train")
axes[1].plot(epochs, history.history["val_loss"], label="validation")
axes[1].set(title="Loss", xlabel="Epoch"); axes[1].legend()
plt.tight_layout()


In [ ]:
# Optional classical models on learned features. These use the same fixed split and
# are not additional independent evaluations.
feature_model = Model(inputs=model.input, outputs=model.get_layer("feature_dense").output)
train_features = feature_model.predict(X_train, verbose=0)
valid_features = feature_model.predict(X_valid, verbose=0)
y_train_int = np.argmax(y_train, axis=1)
y_valid_int = np.argmax(y_valid, axis=1)

svm = SVC(kernel="rbf", random_state=SEED)
svm.fit(train_features, y_train_int)
print("SVM validation accuracy:", svm.score(valid_features, y_valid_int))

boosted = xgb.XGBClassifier(
    n_estimators=100, objective="multi:softprob", eval_metric="mlogloss",
    num_class=len(class_names), random_state=SEED, n_jobs=1,
)
boosted.fit(train_features, y_train_int)
print("XGBoost validation accuracy:", boosted.score(valid_features, y_valid_int))
